**Tabla de contenido**

- [Lectura de datos](#Lectura-de-datos)
- [Preprocesamiento](#Preprocesamiento)
- [Feature engineering](#Feature-engineering)
- [Modelación](#Modelacion)
    - [LSTM Single step](#LSTM-Single-step)

# Lectura de datos

In [ ]:
import pandas as pd
import numpy as np
import os

file_path = lambda file: os.path.join(os.getcwd(),'data/walmart_sales',file)
train_df = pd.read_csv(file_path('train.csv'))
train_df.head(8)

In [ ]:
stores_df = pd.read_csv(file_path('stores.csv'))
stores_df.head(8)

In [ ]:
features_df = pd.read_csv(file_path('features.csv'))
features_df.head(8)

In [ ]:
us_holiday_df = pd.read_csv(file_path('US Holiday Dates (2004-2021).csv'))
us_holiday_df.head(8)

Veamos ahora cual es la fecha máxima y miníma para los datos de entrenamiento. Así mismo, veamos cuantos registros hay en el dataframe de entrenamiento

In [ ]:
print(f"La fecha mínima es: {train_df["Date"].min()} y la máxima es {train_df["Date"].max()}")
print(f"el dataset tiene un total de {len(train_df)} registros")

# Preprocesamiento

Antes de preprocesar los datos, veamos si hay datos faltantes

In [ ]:
train_df.isnull().sum()

Por lo que vemos, no hay datos faltantes. Ahora veamos si hay registros dúplicados.

In [ ]:
duplicados = train_df[train_df.duplicated(subset=['Store','Dept'],keep=False)]
duplicados.head()

Podemos ver que si tenemos datos duplicados. Obtengamos un nuevo dataframe sin registros duplicados y solo conservaremos las columnas Store y Dept

In [ ]:
combinations = train_df[["Store", "Dept"]].drop_duplicates()
combinations

Ahora, enfoquemonos en la columna Date, esta la convertiremos a formato Datetime, mandamos la fecha al index y ordenamos de forma ascendente.

In [ ]:
process_df = train_df.copy()
process_df["Date"] = pd.to_datetime(process_df["Date"], format='%Y-%m-%d')
process_df.set_index("Date", inplace=True)
process_df.sort_index(inplace=True)
process_df.head()

Ahora que ya tenemos la fecha en el index, vamos a remuestrear el dataset de la siguiente forma.

- Rellenar los períodos semanales sin datos de ventas con 0 (asumiendo que no hubo ventas en esas semanas) y asegurar que todas las combinaciones de Store (tienda) + Dept (departamento) tengan un registro semanal continuo (incluso si no hubo ventas en ciertas semanas).

In [ ]:
# Lista para almacenar DataFrames procesados
p_df_resampled_list = []

# Imputación de ventas semanales faltantes por cero
for _, row in combinations.iterrows():
    store_id, dept = row["Store"], row["Dept"]

    # Filtrar por tienda y departamento
    p_df = process_df[(process_df["Store"] == store_id) & (process_df["Dept"] == dept)]

    # Re-muestreo semanal (cada viernes) y completar valores faltantes
    p_df_resampled = p_df.resample('W-FRI').asfreq()

    # Rellenar valores faltantes
    p_df_resampled["Store"] = p_df_resampled["Store"].ffill() # rellena nan hacia adelante con el ultimos valor conocido
    p_df_resampled["Dept"] = p_df_resampled["Dept"].ffill()
    p_df_resampled["Weekly_Sales"] = p_df_resampled["Weekly_Sales"].fillna(0)

    # Agregar a la lista de resultados
    p_df_resampled_list.append(p_df_resampled)


Cón esto ya tenemos el dataframe con datos continuos y sin  huecos garantizando que cada tienda y departamento registre ventas todos los viernes.

In [ ]:
processed_data = pd.concat(p_df_resampled_list)
processed_data = processed_data.reset_index()
processed_data.head()

In [ ]:
print(f"cantidad de registros en el df procesado {processed_data.shape}")
print(f"cantidad de registros en el df original {train_df.shape}")

Ahora, necesitamos que las tiendas y departamentos sean enteros. Así mismo, tranformamos la fecha a formato object 

In [ ]:
processed_data["Store"] = processed_data["Store"].astype(int)
processed_data["Dept"] = processed_data["Dept"].astype(int)
processed_data["Date"] = processed_data["Date"].dt.strftime("%Y-%m-%d")

Veamos si el anterior proceso altero las ventas. Para esto veriquemos que la suma total de ventas semanales en el DataFrame processed_data sea prácticamente igual a la del DataFrame original train_df, con un margen de error muy pequeño (menor a 0.01)

In [ ]:
diff = abs(processed_data["Weekly_Sales"].sum() - train_df["Weekly_Sales"].sum())
print(f"Diferencia absoluta en las sumas de ventas: {diff:.4f}")

assert diff < 0.01

Esto es perfecto. La suma de las ventas procesadas es practicamente igual a la suma de las ventas originales. Entonces podemos decir que train_df = processed_data

In [ ]:
train_df = processed_data

Ahora que ya tenemos un dataframe que contiene las ventas cada viernes. Procedamos a calcular  una fecha de corte. Esto lo vamos a hacer usando el dataframe process_df  que la copia del dataframe original.

- cut_date = max_date - pd.Timedelta(weeks=7, days=1) : esta línea calcula una nueva fecha que es exactamente 7 semanas y 1 día anterior a la fecha almacenada en max_date. 

Este tipo de operación es común cuando necesitas establecer una fecha de corte o umbral en el pasado para filtrar datos

In [ ]:
max_date = max(process_df.index) # Fecha máxima
cut_date = max_date - pd.Timedelta(weeks=7, days=1)
print(f"la última fecha registada es {max_date} y la fecha de corte es {cut_date}")

Ahora que ya conocemos la fecha máxima y la fecha de corte, vamos a :

1. Seleccionar un dataframe que contiene los datos desde la fecha de corte hasta el final de los datos
2. Contar cuántos registros (días/fechas) hay para cada combinación única de Store (tienda) y Dept (departamento) en el dataframe filtrado

In [ ]:
count_train_df = process_df[cut_date:].reset_index() #1
count_train_df = count_train_df.groupby(["Store", "Dept"])["Date"].count().reset_index()
count_train_df.head()

ok. Ahora seleccionemos solo las combinaciones de Store y Dept que tienen al menos 8 registros en el período seleccionado.

In [ ]:
count_train_df = count_train_df[count_train_df["Date"]>=8]
count_train_df = count_train_df[["Store", "Dept"]]
count_train_df.shape

Ahora, vamos a conservar solo las combinaciones de tiendas y departamentos que tienen datos recientes y suficientes registros. Para esto, haremos un merge (unión) entre los datos históricos y la lista de tiendas y departamentos activos después de la fecha de corte

In [ ]:
train_df = train_df.merge(count_train_df, on=["Store", "Dept"], how="inner")
train_df.shape

ok. Podemos ver que de 449237 datos, nos quedamos solo con 406819 registros.

Ahora lo que nos queda hacer es agregar las variables que se encuentran en el dataframe `Stores` al dataframe `train_df`. Estas variables son:
- Tipe
- Size

In [ ]:
print(f"datos antes del merge {train_df.shape}")
merged_train_df = pd.merge(train_df, stores_df, on='Store', how='left')
print(f"Datos después del merge {merged_train_df.shape}")

In [ ]:
merged_train_df.head()

Perfecto!. AHora agregemos las características que hay en el dataframe `Feature_df`. Este merge lo aremos usando las columnas `Store` y `Date`.

In [ ]:
merged_train_df = pd.merge(merged_train_df, features_df.drop(columns="IsHoliday"), on=['Store', 'Date'], how='left')
merged_train_df.head()

Ahora, crearemos una función que ajusta las fechas. En este caso como las ventas deben estar los viernes de cada semana, entonces conviertiremos las fechas de una columna en el viernes anterior o el mismo viernes más cercano hacia atrás.
Además, eliminaremos fechas repetidas después de ese cambio.

1. Si la columna no está en formato fecha, la convertimos usando `pd.to_datetime()
2. Lleva cada fecha al viernes anterior (o al mismo viernes si ya es viernes).
    - `x.weekday()` da el día de la semana (0 = lunes, ..., 4 = viernes).
    - El cálculo `((x.weekday() + (7-4)) % 7)` da cuántos días hay que restar para llegar al viernes más cercano hacia atrás.
        - 7-4 = 3 (viernes es el día 4 en weekday(), pero se usa este ajuste para calcular correctamente el retroceso).
        - Si la fecha es lunes (0): (0 + 3) % 7 = 3 → retrocede 3 días (viernes anterior).
        - Si la fecha es viernes (4): (4 + 3) % 7 = 0 → no retrocede (ya es viernes).
        - Si la fecha es domingo (6): (6 + 3) % 7 = 2 → retrocede 2 días (viernes anterior)
3. Si varias filas terminan con la misma fecha después del ajuste, se eliminan los duplicados.

In [ ]:
def to_previous_friday(df: pd.DataFrame, date_col: str, ) -> pd.DataFrame:
    if not isinstance(df[date_col][0], pd.Timestamp): #1
        df[date_col] = pd.to_datetime(df[date_col])

    df[f"{date_col}"] = df[date_col].apply(lambda x: x - pd.DateOffset(days=((x.weekday() + (7-4)) % 7))) # 2
    df = df.drop_duplicates(subset=[date_col]) #3
    return df

Ahora al dataframe `us_holiday_df le hacemos lo siguiente:
1. Aplicamos la función creada para ajustar la fecha al viernes anterior
2. Creamos una nueva columna llamada IsHoliday que copia los valores de la columna Holiday
3. Seleccionar columnas de interes

In [ ]:
us_holiday_df = to_previous_friday(us_holiday_df, "Date") #1
us_holiday_df["IsHoliday"] = us_holiday_df["Holiday"] #2
us_holiday_df = us_holiday_df[["Date", "IsHoliday"]] #3
us_holiday_df.head()

Ahora, crearemos un función para agregar la fecha como variable predictora al dataset.

In [ ]:
def add_more_date_feature(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    if not isinstance(df[date_col][0], pd.Timestamp):
        df[date_col] = pd.to_datetime(df[date_col])
    
    df["year"] = df[date_col].apply(lambda x: x.year)
    df["quarter"] = df[date_col].apply(lambda x: x.quarter)
    df["month"] = df[date_col].apply(lambda x: x.month)
    df["week"] = df[date_col].apply(lambda x: x.week)

    return df

Ahora, apliquemos la función al dataframe `merged_train_df` para agregar la fecha como variable predictora.

In [ ]:
merged_train_df = add_more_date_feature(merged_train_df, "Date")

Ahora realicemos el `merge` entre los dataframe `merged_train_df` y `us_holiday_df`, la columna en común va a ser `Date` y usaremos como columna de festivos la que está en el dataframe `us_holiday_df`

In [ ]:
merged_train_df = pd.merge(merged_train_df.drop(columns="IsHoliday"), us_holiday_df, on="Date", how="left")
merged_train_df.head()

# Feature engineering

En esta sección nos enfocaremos en:
- Codificación de variables categóricas
- Tratamiento de valores faltantes

Pero antes de eso, veamos los valores faltantes que tenemos

In [ ]:
nan_sum = {col:merged_train_df[col].isnull().sum() for col in merged_train_df.columns}
nan_sum = pd.DataFrame.from_dict(nan_sum, orient='index',columns=['nulos'])
nan_sum.head(len(nan_sum))

OK. Tenemos varias columnas con valores faltantes. Procedamos a codificar las variables categóricas.

In [ ]:
type_group = {'A': 3, 'B': 2, 'C': 1}

merged_train_df['Type'] = merged_train_df['Type'].map(type_group)

Ahora codifiquemos en variables dummy los días festivos.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(sparse_output=False)
onehot_encoded = encoder.fit_transform(merged_train_df[["IsHoliday"]])
onehot_encoded_df = pd.DataFrame(onehot_encoded, columns=encoder.get_feature_names_out(["IsHoliday"]))
onehot_encoded_df.head()

Ok. con los dias festivos codificados a variables dummy, procedemos a concatenarla al  dataframe merged_train_df

In [ ]:
merged_train_df = pd.concat([merged_train_df, onehot_encoded_df], axis=1)
merged_train_df = merged_train_df.drop(columns=["IsHoliday", "IsHoliday_nan"], axis=1)
merged_train_df.head()

Ahora que ya tenemos las variables categóricas codificadas en dummy. Los valores faltantes los vamos a poner en cero.

In [ ]:
mrkdw_cols = ["MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5", "Weekly_Sales"]
merged_train_df[mrkdw_cols] = merged_train_df[mrkdw_cols].where(merged_train_df[mrkdw_cols]>=0, 0)

In [ ]:
merged_train_df.head()

Vamos crear una función que nos permite crear características temporales rezagadas (lags), respetando el orden temporal y los grupos (si existen), y permite imputar los valores iniciales de forma flexible. 

In [ ]:
from typing import Union, List
def feature_lag_values(df: pd.DataFrame, sort_col: str, col: str, lag: list, grp_col: Union[str, List] = None, impute_method: str = "zeros") -> pd.DataFrame:
    if grp_col:
        for n in lag:
            df[f"{col}_lag{n}"] = df.sort_values(by=sort_col).groupby(grp_col)[col].shift(n)
    else:
        for n in lag:
            df[f"{col}_lag{n}"] = df.sort_values(by=sort_col)[col].shift(n)
    
    if impute_method == "zeros":
        for n in lag:
            df[f"{col}_lag{n}"] = df[f"{col}_lag{n}"].fillna(0)
    else:
        for n in lag:
            df[f"{col}_lag{n}"] = df[f"{col}_lag{n}"].fillna(np.NaN)

    return df

sort_col = "Date"
grp_col = ["Store", "Dept"]
cols = ["Weekly_Sales"]
lags = [1,2]

for col in cols:
    merged_train_df = feature_lag_values(merged_train_df, sort_col, col, lags, grp_col)
    # df = feature_hist_agg_values(df, sort_col, col, lags, func, grp_col)

merged_train_df

Veamos sin hay valores faltantes

In [ ]:
merged_train_df.isnull().sum()

# Modelacion

Los modelos que vamos usar son los siguientes:

1) SARIMA 
2) Elastic Net 
3) RF (all)
4) LGBM (all)
5) LSTM/AR LSTM 
6) GRU 

Pero antes de continuar, organicemos los nombres de las columnas.

In [ ]:
import re
merged_train_df = merged_train_df.rename(columns = lambda x:re.sub('[^A-Za-z0-9_]+', '', x))
merged_train_df.columns

Ahora preparemos el dataframe de series de tiempo para ser divido usando validación cruzada temporal (TimeSeriesSplit) de de scikit-learn.

1. Se importa `TimeSeriesSplit`, que permite realizar validación cruzada respetando el orden temporal (es decir, sin usar datos futuros para predecir el pasado).
2. Configurar el número de divisiones y separación entre ellas. 
    - `n_splits=2`: se harán 2 divisiones, produciendo 2 conjuntos de entrenamiento/validación secuenciales.
    - `gap=0`: no se deja espacio (gap) entre el conjunto de entrenamiento y el de validación.
3. Esto define una fecha de corte manual
4. Esto indica cuántos pasos atrás (lags) se usan como características temporales. Por ejemplo, si predices sales, podrías usar sales_lag1 y sales_lag2.
5. La fecha se agrega al index
6. Se ordena el indice por fechas
7. Calcular la "fecha base" para evitar usar lags inválidos:
    - min(df.index): obtiene la fecha más temprana del DataFrame.

    - pd.Timedelta(weeks=max_lags): calcula un intervalo de tiempo igual al número de lags, en semanas.

    - base_date: es la fecha más temprana desde la cual ya puedes calcular los lags correctamente.

    Por ejemplo, si tienes un rezago de 2 semanas, no puedes entrenar un modelo con fechas anteriores porque no tendrás lag1 y lag2


In [ ]:
from sklearn.model_selection import TimeSeriesSplit #1

tss = TimeSeriesSplit(n_splits = 2, gap=0) #2
train_cut_date = "2012-10-01" #3
max_lags  = 2 #4
df = merged_train_df.copy()

df.set_index("Date", inplace=True) #5
df.sort_index(inplace=True) #6
# base date > data will be split
base_date = min(df.index) + pd.Timedelta(weeks=max_lags) #7.
print(base_date)

hora que conocemos la fecha base, hacemos lo siguiente:
1. Eliminar las fechas no aptas por los lags: Esto recorta el DataFrame desde base_date en adelante. Recuerda que base_date = min(df.index) + pd.Timedelta(weeks=max_lags), así que esta línea elimina las primeras filas, que no tendrían valores válidos de lag (por falta de datos previos).


In [ ]:
df = df[base_date:] #1
# dividir en entrenamiento y test
train_cut_date = "2012-10-01" 
tr_df = df[:train_cut_date]
te_df = df[train_cut_date:]

# separamos en variables de entrenamiento y objetivo
tgt_col = "Weekly_Sales"
X_train = tr_df.drop(columns=tgt_col, axis=1)
y_train = tr_df[tgt_col]

X_test = te_df.drop(columns=tgt_col, axis=1)
y_test = te_df[tgt_col]

X_train.head()

In [ ]:
print(X_train.index)

Veamos ahora los datos.

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(12,6))
y_train.groupby("Date").sum().plot(label="train")
y_test.groupby("Date").sum().plot(label="test")
plt.legend()
plt.title("Data split")
plt.show()

Ahora que ya tenemos los datos, es momento de escalar los datos , en este caso utilizaremos MinMaxScales que escala los datos a un rango arbitrario, por defecto es (0,1), y lo hace restando el valor mínimo a cada componente del vector o columna y dividiendo entre la diferencia entre el valor máximo y el mínimo.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(0, 1))
scaler1 = MinMaxScaler(feature_range=(0, 1))

tr_df_sc = tr_df.copy()
# definamos las columnas a escalar
sc_col = ['Type', 'Size', 'Temperature', 'Fuel_Price', 
          'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 
          'MarkDown5', 'CPI', 'Unemployment', 'year', 'quarter', 
          'month', 'week'] 
tr_df_sc[sc_col] = scaler.fit_transform(tr_df_sc[sc_col])  # Escala las características
tr_df_sc[["Weekly_Sales"]] = scaler1.fit_transform(tr_df_sc[["Weekly_Sales"]]) # Escala la variable objetivo
tr_df_sc.head()

Ahora, vamos a creear una función que nos devolverá las secuencias y un diccionario que contiene la información correspondiente a cada valor Y que se utlizó en las secuencias como objetivo. Esta función agrupa los datos por `Store` y `Dept`, cada grupo es un un dataframe al que se le sacan las secuencias. En este caso usaremos secuencias de datos de 2 registros.

- `Secuencia`: Cada secuencia es una lista que contiene tanto las `features` como el `target`, con la estructura [features, target]. Aquí, `features es una matriz de dimensión (n, n)`, mientras que target es un valor escalar.

Un aspecto importante a destacar en esta función es que todas las secuencias generadas tendrán la misma longitud. Esto se debe a que la ventana se desliza utilizando el índice [i : i + time_step]. En cada iteración, i se incrementa en una unidad, y la longitud de cada secuencia está determinada por el valor de time_step.

Por ejemplo, si tenemos una lista de datos:

datos = [10, 20, 30, 40, 50, 60]
time_step = 3

Entonces, al aplicar la lógica de la ventana deslizante [i : i + time_step], las secuencias generadas serán:

datos[0:3] → [10, 20, 30] →  target [40]

datos[1:4] → [20, 30, 40] →  target [50]

datos[2:5] → [30, 40, 50] →  target [60]


In [ ]:
def crear_secuencia(data, time_steps,drop_cols):
    sequecias = []
    output_list =[]
    for seq, df  in data.groupby(["Store","Dept"]):
        process_df = df.reset_index(drop=True)               # Reseteamos el index del dataframe
        features = process_df.drop(columns=drop_cols).values # Eliminamos las columnas que no se usaran como entradas
        targets = process_df["Weekly_Sales"].values           # Columna objetivo
        # ciclo for para crear las secuencias
        for i in range(len(df)-time_steps):
            feature = features[i:(i+time_steps)]             # hacemos un desplazamiento de la posición actual i + time_step
            target = targets[i+time_steps]                   # el objetivo será el escalar qué esté en la posición i + time_steps
            sequecias.append((feature,target))               # Guarda la secuencia y el target en una lista
            output_list.append({
                'Date':df.index[i+time_steps],
                'Store':df['Store'].iloc[i+time_steps],
                'Dept': df['Dept'].iloc[i+time_steps],
                'Weekly_Sales': df['Weekly_Sales'].iloc[i+time_steps]
            })
    return sequecias, output_list

Cómo ya tenemos la función, procedamos a crear las secuencias.

In [ ]:
time_steps = 2  # para la creación de la secuencia solo avanzará dos pasos
drop_cols = ["Store", "Dept", "Weekly_Sales_lag1", "Weekly_Sales_lag2"]
sequences, output_list = crear_secuencia(tr_df_sc, time_steps, drop_cols)
output = pd.DataFrame(output_list)

In [ ]:
output.head()

Veamos ahora como estan los datos en lista de secuencias

In [ ]:
for features, target in enumerate(sequences[1]):
    print(f'Secuencia{features}')
    print(f'objetivo {target}')

Perfecto, ahora lo que necesitamos es convertir estas secuencias a la forma correcta en que las redes LSTM las reciben. En este caso es `(total_registros, longitud de la secuencia, características)`

In [ ]:
X,y = zip(*sequences)
X = np.array(X)
y = np.array(y)
print(X.shape)
print(y.shape)

# LSTM Single step

Definamos la función que entrenará el modelo

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.losses import MeanSquaredError,MeanAbsoluteError
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import PolynomialDecay

def compilar_y_entrenar(modelo,patience = 5, max_epochs=50):
    # parada temprana
    early_stop = EarlyStopping(
        monitor ='val_loss',
        patience = patience,
        restore_best_weights = True,
        mode = 'min' # vamos a minimizar el mae
    )

    # compilar el modelo
    modelo.compile(
        loss = MeanSquaredError(), # función de pérdida MSE
        optimizer = Adam(),
        metrics = [MeanAbsoluteError()] # función de evalución MAE
    )
    # entrenar modelo
    history = modelo.fit(
        x = X,
        y = y,
        batch_size=32,
        validation_split = 0.05, # usamos el 5% para validar
        epochs = max_epochs,
        callbacks =[early_stop]
    )
    return history

Ahora que ya tenemos la función creada, es momento de crear el modelo

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout


tf.keras.backend.clear_session() # reiniciar el kernel
lSTM_model = Sequential([
    LSTM(128,return_sequences =True), # usa valores del pasado
    Dropout(0.5),
    LSTM(128),
    Dropout(0.5),
    Dense(units =1)
])

In [ ]:
history = compilar_y_entrenar(lSTM_model)